# 01 — Setup & Data Audit

**Purpose:** verify the environment, mount Drive, and run a **hard validation** on the dataset.

This notebook *will hard-fail* if any class has fewer than 5 test images, or if any class is missing from any split. If validation fails, run `02_split_dataset.ipynb` to re-split, then come back here.

Outputs:
- `results/metrics/dataset_audit.json` — committed to GitHub.

## 1. Environment setup

This notebook is designed for **Google Colab (A100)**. The setup cell below:

1. Mounts Google Drive
2. Clones or updates the GitHub repo
3. Installs requirements
4. Adds the repo root to `sys.path` so `from src...` works

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/musarashid49/Image-Classification-with-CNN.git"
REPO_DIR = Path("/content/Image-Classification-with-CNN")

# 1. Mount Drive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab — skipping drive mount.")

# 2. Clone or pull
if REPO_DIR.exists():
    print(f"Repo already cloned at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    print(f"Cloning {REPO_URL} -> {REPO_DIR}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# 3. Make repo importable
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# 4. Install requirements (Colab usually has torch already)
req = REPO_DIR / "requirements.txt"
if req.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already cloned at /content/Image-Classification-with-CNN; pulling latest...
cwd: /content/Image-Classification-with-CNN
sys.path[0]: /content/Image-Classification-with-CNN


## 2. Imports & paths

In [2]:
from pathlib import Path
import json
from PIL import Image

from config.config import (
    CLASS_NAMES, NUM_CLASSES,
    DATASET_DIR, DATASET_DIR_LEGACY,
    MIN_IMAGES_PER_CLASS, MIN_TEST_IMAGES_PER_CLASS,
    LOCAL_METRICS,
)
from src.dataset import count_images_per_class
from src.utils import ensure_dir, write_json, get_device, gpu_info

print(f"Device: {get_device()}")
print(f"GPU: {gpu_info() or 'CPU only'}")
print(f"Expected dataset: {DATASET_DIR}")
print(f"Number of classes: {NUM_CLASSES}")

Device: cuda
GPU: NVIDIA A100-SXM4-40GB (42.4 GB)
Expected dataset: /content/drive/MyDrive/dataset_resplit
Number of classes: 16


In [3]:
# ── DIAGNOSTIC: show actual folder names vs config CLASS_NAMES ──────────────
from config.config import DATASET_DIR, CLASS_NAMES

train_dir = DATASET_DIR / "train"
actual = sorted(p.name for p in train_dir.iterdir() if p.is_dir())

config_set  = set(CLASS_NAMES)
actual_set  = set(actual)

print(f"Actual folders on Drive ({len(actual)}):")
for name in actual:
    match = "✓" if name in config_set else "✗  ← NOT in config"
    print(f"  {name:<40s} {match}")

print()
print(f"In config but NOT on Drive ({len(config_set - actual_set)}):")
for name in sorted(config_set - actual_set):
    print(f"  {name}")

Actual folders on Drive (16):
  ahmed_sharif_chaudhry                    ✓
  altaf_hussain                            ✓
  asfandyar_wali                           ✓
  asif_ali_zardari                         ✓
  bilawal_bhutto                           ✓
  chaudhry_nisar                           ✓
  fazlur_rehman                            ✓
  imran_khan                               ✓
  maryam_nawaz                             ✓
  nawaz_sharif                             ✓
  pervez_khattak                           ✓
  pervez_musharraf                         ✓
  rana_sanaullah                           ✓
  shah_mehmood_qureshi                     ✓
  shehbaz_sharif                           ✓
  sirajul_haq                              ✓

In config but NOT on Drive (0):


## 3. Sanity check: dataset folder exists

In [4]:
if not DATASET_DIR.exists():
    print(f"⚠️  {DATASET_DIR} not found.")
    if DATASET_DIR_LEGACY.exists():
        print(f"   Legacy dataset exists at {DATASET_DIR_LEGACY}.")
        print(f"   👉 Run notebook 02_split_dataset.ipynb to produce {DATASET_DIR.name}.")
    raise FileNotFoundError(f"Dataset not found at {DATASET_DIR}")
print(f"✓ dataset directory exists: {DATASET_DIR}")

✓ dataset directory exists: /content/drive/MyDrive/dataset_resplit


## 4. Per-class image counts

In [5]:
counts = count_images_per_class(DATASET_DIR)
totals = {s: sum(c.values()) for s, c in counts.items()}
grand_total = sum(totals.values())

print(f"{'class':<32s} {'train':>6s} {'val':>6s} {'test':>6s} {'total':>6s}")
print("-" * 64)
for cls in CLASS_NAMES:
    tr = counts['train'].get(cls, 0)
    va = counts['val'].get(cls, 0)
    te = counts['test'].get(cls, 0)
    tot = tr + va + te
    flag = "  ⚠️" if (te < MIN_TEST_IMAGES_PER_CLASS or tot < MIN_IMAGES_PER_CLASS) else ""
    print(f"{cls:<32s} {tr:>6d} {va:>6d} {te:>6d} {tot:>6d}{flag}")

print("-" * 64)
print(f"{'TOTAL':<32s} {totals['train']:>6d} {totals['val']:>6d} {totals['test']:>6d} {grand_total:>6d}")

class                             train    val   test  total
----------------------------------------------------------------
ahmed_sharif_chaudhry                60     12      8     80
altaf_hussain                        77     15     11    103
asfandyar_wali                       72     14     11     97
asif_ali_zardari                     56     11      8     75  ⚠️
bilawal_bhutto                       53     10      8     71  ⚠️
chaudhry_nisar                       73     14     11     98
fazlur_rehman                        87     17     13    117
imran_khan                           67     13     10     90
maryam_nawaz                         93     18     14    125
nawaz_sharif                         54     10      8     72  ⚠️
pervez_khattak                       53     10      8     71  ⚠️
pervez_musharraf                     83     16     12    111
rana_sanaullah                       54     10      9     73  ⚠️
shah_mehmood_qureshi                 66     13     10     89


## 5. Hard validation — fails if dataset is unfit

In [6]:
errors = []
warnings = []

# 5a. All 16 classes present in each split
for split in ('train', 'val', 'test'):
    for cls in CLASS_NAMES:
        if cls not in counts[split]:
            errors.append(f"class '{cls}' missing from split '{split}'")

# 5b. Test set per-class minimum
for cls in CLASS_NAMES:
    n_test = counts['test'].get(cls, 0)
    if n_test < MIN_TEST_IMAGES_PER_CLASS:
        errors.append(f"class '{cls}' has only {n_test} test images "
                      f"(< {MIN_TEST_IMAGES_PER_CLASS} required)")

# 5c. Total per-class minimum
for cls in CLASS_NAMES:
    tot = sum(counts[s].get(cls, 0) for s in ('train', 'val', 'test'))
    if tot < MIN_IMAGES_PER_CLASS:
        warnings.append(f"class '{cls}' has only {tot} total images "
                        f"(< {MIN_IMAGES_PER_CLASS} preferred)")

if errors:
    print("❌ VALIDATION FAILED:")
    for e in errors:
        print(f"   - {e}")
    print()
    print("👉 Run notebook 02_split_dataset.ipynb to fix.")
    raise RuntimeError("Dataset validation failed — fix before training.")

if warnings:
    print("⚠️  Soft warnings (training will proceed):")
    for w in warnings:
        print(f"   - {w}")
print("\n✓ dataset validation passed.")

⚠️  Soft warnings (training will proceed):
   - class 'asif_ali_zardari' has only 75 total images (< 80 preferred)
   - class 'bilawal_bhutto' has only 71 total images (< 80 preferred)
   - class 'nawaz_sharif' has only 72 total images (< 80 preferred)
   - class 'pervez_khattak' has only 71 total images (< 80 preferred)
   - class 'rana_sanaullah' has only 73 total images (< 80 preferred)
   - class 'sirajul_haq' has only 70 total images (< 80 preferred)

✓ dataset validation passed.


## 6. Leakage check — no filename appears in more than one split

If the same image filename appears in two splits, evaluation metrics are invalid.

In [8]:
# ── Cell 6: Leakage check — same (class, filename) in more than one split ───
filenames_by_split = {}
for split in ("train", "val", "test"):
    pairs = set()
    for cls in CLASS_NAMES:
        d = DATASET_DIR / split / cls
        if d.exists():
            for p in d.iterdir():
                if p.is_file():
                    pairs.add((cls, p.name))   # (class, filename) tuple
    filenames_by_split[split] = pairs

tr_va = filenames_by_split["train"] & filenames_by_split["val"]
tr_te = filenames_by_split["train"] & filenames_by_split["test"]
va_te = filenames_by_split["val"]   & filenames_by_split["test"]

if tr_va or tr_te or va_te:
    print("❌ LEAKAGE DETECTED (same image in same class across splits):")
    if tr_va:
        print(f"   train ∩ val:  {len(tr_va)} files, e.g. {list(tr_va)[:3]}")
    if tr_te:
        print(f"   train ∩ test: {len(tr_te)} files, e.g. {list(tr_te)[:3]}")
    if va_te:
        print(f"   val ∩ test:   {len(va_te)} files, e.g. {list(va_te)[:3]}")
    raise RuntimeError("Real data leakage between splits — re-run 02_split_dataset.ipynb.")
print("✓ no data leakage between splits.")

✓ no data leakage between splits.


## 7. Corrupt image scan

In [9]:
bad_images = []
for split in ('train', 'val', 'test'):
    for cls in CLASS_NAMES:
        d = DATASET_DIR / split / cls
        if not d.exists():
            continue
        for p in d.iterdir():
            if not p.is_file():
                continue
            try:
                with Image.open(p) as im:
                    im.verify()
            except Exception as e:
                bad_images.append({"path": str(p), "error": str(e)})

if bad_images:
    print(f"⚠️  found {len(bad_images)} corrupt images:")
    for b in bad_images[:10]:
        print(f"   {b['path']}: {b['error']}")
    print("   Consider deleting these before training.")
else:
    print("✓ no corrupt images detected.")

✓ no corrupt images detected.


## 8. Save the audit report

In [10]:
audit = {
    "dataset_dir": str(DATASET_DIR),
    "totals": totals,
    "grand_total": grand_total,
    "per_class_counts": counts,
    "errors": errors,
    "warnings": warnings,
    "corrupt_images": bad_images,
    "validation_passed": len(errors) == 0,
}
out_path = LOCAL_METRICS / "dataset_audit.json"
write_json(audit, out_path)
print(f"✓ audit saved to {out_path}")
print()
print("Next: run 03_train_model_A.ipynb")

✓ audit saved to /content/Image-Classification-with-CNN/results/metrics/dataset_audit.json

Next: run 03_train_model_A.ipynb
